In [ ]:
import sys; sys.path.append('..')
import inflation, pickle, wall_generation, MeshFEM, mesh_utilities, mesh, parametrization
import numpy as np, wall_width_formulas as wwf, visualization
from matplotlib import pyplot as plt

In [ ]:
mkdir -p rendering_data

In [ ]:
tsurf = mesh_utilities.subdivide_loop(mesh.Mesh('../../examples/squidward_remesh.obj'), 1)

In [ ]:
tsurf.save('rendering_data/target_surface.obj')

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(tsurf, np.loadtxt('../results/squidward/param_laptop.txt.gz'))

In [ ]:
param_mfw = mesh.MSHFieldWriter('rendering_data/parametrization_fields.msh', rparam.uv(), tsurf.triangles())
phi = rparam.leftStretchAngles()
param_mfw.addField('contract_dir_angle', phi)
param_mfw.addField('contract_dir', np.hstack([np.cos(phi)[:, np.newaxis], np.sin(phi)[:, np.newaxis]]))
param_mfw.addField('tube_dir', np.hstack([np.cos(phi + np.pi / 2)[:, np.newaxis], np.sin(phi + np.pi / 2)[:, np.newaxis]]))
param_mfw.addField('tube_dir_angle', phi + np.pi / 2)
param_mfw.addField('alpha', rparam.getAlphas())
del param_mfw

In [ ]:
nsubdiv = 3
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)

alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(3, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)

(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=3.75)

visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf)
plt.savefig('rendering_data/strip_field_sdf.png')
plt.close()
smfw = mesh.MSHFieldWriter('rendering_data/stripe_field_sdf.msh', sdfVertices, sdfTris)
smfw.addField('wall_sdf', sdf)

In [ ]:
targetEdgeSpacing = 0.25
minContourLen = 0.75

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=targetEdgeSpacing,
                                              minContourLen=minContourLen)

In [ ]:
visualization.plot_line_segments(pts, edges, width=20, height=16)
plt.savefig('rendering_data/initial_walls.svg')
plt.close()

In [ ]:
from sheet_meshing import generateSheetMesh
s, iwv, iwbv = generateSheetMesh(sdfVertices, sdfTris, sdf, triArea=0.05, permitWallInteriorVertices=False, targetEdgeSpacing=targetEdgeSpacing)

In [ ]:
import os
os.system('scp fidis:/scratch/fpanetta/with_smoothing/squidward/fw_0.01/smoothingWeight_0.01/fixBdry_1/triArea_0.05/edgeSpacing_0.25/cb_0.2/equilibrium_nofit_reinflate.msh rendering_data/fixed_bdry_opt_inflated.msh')
os.system('scp fidis:/scratch/fpanetta/with_smoothing/squidward/fw_0.01/smoothingWeight_0.01/fixBdry_1/triArea_0.05/edgeSpacing_0.25/cb_0.2/design_post_lowfit_opt.msh rendering_data/fixed_bdry_opt_design.msh')
os.system('scp fidis:/scratch/fpanetta/with_smoothing/squidward/fw_0.01/smoothingWeight_0.01/fixBdry_0/triArea_0.05/edgeSpacing_0.5/cb_0.4/equilibrium_nofit_reinflate.msh rendering_data/free_bdry_opt_inflated.msh')
os.system('scp fidis:/scratch/fpanetta/with_smoothing/squidward/fw_0.01/smoothingWeight_0.01/fixBdry_0/triArea_0.05/edgeSpacing_0.5/cb_0.4/design_post_lowfit_opt.msh rendering_data/free_bdry_opt_design.msh')

In [ ]:
import os
os.system('scp -r fidis:/scratch/fpanetta/with_smoothing/squidward/fw_0.01/smoothingWeight_0.01/fixBdry_1/triArea_0.05/edgeSpacing_0.25/cb_0.2/inflation_animation rendering_data/opt_inflation_fixed')
os.system('scp -r fidis:/scratch/fpanetta/with_smoothing/squidward/fw_0.01/smoothingWeight_0.01/fixBdry_0/triArea_0.05/edgeSpacing_0.5/cb_0.4/inflation_animation rendering_data/opt_inflation_free')

In [ ]:
import glob
for f in list(glob.glob('rendering_data/*_bdry_opt_inflated.msh')) + list(glob.glob('rendering_data/*/*.msh')):
    mesh.Mesh(f).save(f.replace('.msh', '.obj'))
    os.unlink(f)

In [ ]:
isheet = inflation.InflatableSheet(s, iwbv)
bv = isheet.mesh().boundaryVertices()
bdryVars = [isheet.varIdx(0, i, c) for i in bv for c in range(3)]

uv = rparam.uv()
paramSampler = mesh_utilities.SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), tsurf.triangles())
liftedSheetPositions = paramSampler.sample(s.vertices(), tsurf.vertices())

In [ ]:
harmonicPos = parametrization.harmonic(isheet.mesh(), liftedSheetPositions[bv])
isheet.setUninflatedDeformation((0.85 * harmonicPos + 0.15 * liftedSheetPositions).transpose(), prepareRigidMotionPinConstraints=False)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(visualization.EquilibriumMesh(isheet))
viewer.show()

In [ ]:
import py_newton_optimizer
niter = 5000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-6

In [ ]:
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, tsurf)
targetAttractedSheet.fittingWeight = 1e-10

In [ ]:
mkdir -p rendering_data/orig_inflation_fixed

In [ ]:
mkdir -p rendering_data/orig_inflation_free

In [ ]:
import time
inflation.benchmark_reset()
opts.niter = iterations_per_output
isheet.pressure = 0.25
#fixedVars = bdryVars
fixedVars = []
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts)
    #isheet.visualizationMesh().save(f'rendering_data/orig_inflation_fixed/step_{step}.obj')
    isheet.visualizationMesh().save(f'rendering_data/orig_inflation_free/step_{step}.obj')
    viewer.update()
    if (cr.numIters() < iterations_per_output): break
inflation.benchmark_report()